# NeuroSegment-BraTS: Treinamento do Modelo (Deep Learning)

**Notebook 4:** Modelagem e Experimentação

Nesta etapa, instanciamos a arquitetura da rede neural, definimos os parâmetros de otimização e executamos o ciclo de treinamento.

**Metodologia Experimental:**
*   **Modelo:** U-Net 3D (arquitetura padrão ouro para segmentação médica volumétrica).
*   **Função de Perda:** `DiceCELoss`. Uma combinação da *Cross-Entropy* (para classificação geral) com a *Dice Loss* (para otimizar o ganho sobre o desbalanceamento das classes tumorais).
*   **Otimizador:** Adam, que se ajusta bem a gradientes esparsos.
*   **Gestão de Hardware:** Como o sistema possui restrições de memória (16GB RAM/VRAM), o treinamento será feito em "Patches" (blocos menores de 96x96x96 gerados no *DataLoader*) e a validação será reconstruída via `SlidingWindowInferer`.

## 1. Importações e Inicialização

In [5]:
import os
import torch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss
from monai.metrics import DiceMetric
from monai.inferers import sliding_window_inference
import matplotlib.pyplot as plt

# Definindo o dispositivo de aceleração (CUDA/GPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Treinamento rodando no dispositivo: {device}")

# Limpeza preventiva do Cache da GPU
if device.type == 'cuda':
    torch.cuda.empty_cache()

Treinamento rodando no dispositivo: cuda


## 2. Ingestão do Pipeline de Dados
Abaixo, conectamos este notebook aos iteradores criados no Notebook 3.

In [6]:
import sys
sys.path.append('../')
from src.data_pipeline import get_dataloaders

# 1. Ingestão do Pipeline
train_loader, val_loader = get_dataloaders()
print("DataLoaders carregados com sucesso no Notebook 4!")

# Validando a conexão (assumindo que train_loader e val_loader estão definidos no kernel)
test_batch = next(iter(train_loader))
print(f"Shape de entrada da imagem: {test_batch['image'].shape}") 
# Esperado: (Batch, Canais, X, Y, Z) -> ex: (2, 4, 96, 96, 96)

Total de pacientes carregados na tubulação: 368
DataLoaders carregados com sucesso no Notebook 4!
Shape de entrada da imagem: torch.Size([4, 4, 96, 96, 96])


## 3. Definição da Arquitetura (U-Net 3D) e Hiperparâmetros
Configuramos a U-Net para aceitar 4 canais de entrada (FLAIR, T1, T1ce, T2) e devolver predições para as nossas 4 classes de interesse (0, 1, 2, 4).

In [7]:
# 1. O Modelo: U-Net Tridimensional
model = UNet(
    spatial_dims=3,             # Imagem 3D
    in_channels=4,              # 4 Modalidades (FLAIR, T1, T1ce, T2)
    out_channels=4,             # 4 Classes de saída (Fundo/Normal, NCR, ED, ET)
    channels=(16, 32, 64, 128, 256), # Número de filtros por camada (mantido leve por causa dos 16GB de VRAM)
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)

# 2. A Função de Perda (Loss Function)
# Combina Softmax (CrossEntropy) com a métrica espacial (Dice)
loss_function = DiceCELoss(to_onehot_y=True, softmax=True)

# 3. O Otimizador
learning_rate = 1e-4
optimizer = torch.optim.Adam(model.parameters(), learning_rate)

# 4. A Métrica de Avaliação Oficial (Dice Score)
# Mede a sobreposição entre a máscara da IA e o gabarito do médico
dice_metric = DiceMetric(include_background=False, reduction="mean")

## 4. O Ciclo de Treinamento e Validação
Executamos o treinamento iterativo. A cada *batch*, o modelo tenta segmentar o tumor, calcula o erro e ajusta seus pesos. Ao final de cada época, rodamos uma etapa de validação (sem aprender) para medir a performance real.

In [ ]:
# Configurações do Loop
max_epochs = 100  # PARA TESTE. Aumente para 50+ na versão final
val_interval = 4 # Roda validação a cada 2 épocas

best_metric = -1
best_metric_epoch = -1
epoch_loss_values = []
metric_values = []

print("INICIANDO O TREINAMENTO...")

for epoch in range(max_epochs):
    print(f"\n--- Época {epoch + 1}/{max_epochs} ---")
    
    # === MODO DE TREINO ===
    model.train()
    epoch_loss = 0
    step = 0
    
    for batch_data in train_loader:
        step += 1
        # Enviando dados para a GPU
        inputs = batch_data["image"].to(device)
        labels = batch_data["label"].to(device)
        
        # O Ciclo do Gradiente
        optimizer.zero_grad()       # Limpa o "lixo" anterior
        outputs = model(inputs)     # Predição (Aposta da IA)
        loss = loss_function(outputs, labels) # Cálculo do Erro
        loss.backward()             # Propagação para trás
        optimizer.step()            # Ajuste dos pesos
        
        epoch_loss += loss.item()
        
        if step % 10 == 0: # Imprime o status a cada 10 lotes processados
            print(f"Lote {step}/{len(train_loader)} | Loss do Lote: {loss.item():.4f}")
            
    epoch_loss /= step
    epoch_loss_values.append(epoch_loss)
    print(f"--> Loss Médio da Época {epoch + 1}: {epoch_loss:.4f}")
    
    # === MODO DE VALIDAÇÃO ===
    if (epoch + 1) % val_interval == 0:
        model.eval()
        with torch.no_grad(): # Desliga o aprendizado para economizar memória
            for val_data in val_loader:
                val_inputs = val_data["image"].to(device)
                val_labels = val_data["label"].to(device)
                
                # A Reconstrução (Sliding Window):
                # Como a imagem de validação é o cérebro inteiro e não cabe na VRAM,
                # o MONAI "escorrega" uma janela de 96x96x96 por todo o cérebro, processa os pedaços e cola tudo no final.
                val_outputs = sliding_window_inference(
                    inputs=val_inputs, 
                    roi_size=(96, 96, 96), 
                    sw_batch_size=4, 
                    predictor=model
                )
                
                # Formata a saída (Probabilidades para Classes Discretas)
                val_outputs = [torch.argmax(i, dim=0, keepdim=True) for i in val_outputs]
                
                # Calcula a nota (Dice Score)
                dice_metric(y_pred=val_outputs, y=val_labels)
            
            # Tira a média da nota de todos os pacientes da validação
            metric = dice_metric.aggregate().item()
            dice_metric.reset()
            metric_values.append(metric)
            
            print(f">>> Métrica Dice de Validação: {metric:.4f}")
            
            # Salvando o modelo se for o novo recorde
            if metric > best_metric:
                best_metric = metric
                best_metric_epoch = epoch + 1
                torch.save(model.state_dict(), os.path.join("../models", "best_metric_model_Unet_3D.pth"))
                print("🏆 Novo recorde! Modelo salvo no disco.")

print(f"\nTREINAMENTO CONCLUÍDO! Melhor Dice: {best_metric:.4f} na época {best_metric_epoch}")

INICIANDO O TREINAMENTO...

--- Época 1/100 ---
Lote 10/328 | Loss do Lote: 2.2137
Lote 20/328 | Loss do Lote: 2.0994
Lote 30/328 | Loss do Lote: 2.0461
Lote 40/328 | Loss do Lote: 2.0486
Lote 50/328 | Loss do Lote: 1.9471
Lote 60/328 | Loss do Lote: 1.9452
Lote 70/328 | Loss do Lote: 1.9216
Lote 80/328 | Loss do Lote: 1.8363
Lote 90/328 | Loss do Lote: 1.8548
Lote 100/328 | Loss do Lote: 1.7525
Lote 110/328 | Loss do Lote: 1.8020
Lote 120/328 | Loss do Lote: 1.7116
Lote 130/328 | Loss do Lote: 1.6838
Lote 140/328 | Loss do Lote: 1.5378
Lote 150/328 | Loss do Lote: 1.5929
Lote 160/328 | Loss do Lote: 1.5788
Lote 170/328 | Loss do Lote: 1.4833
Lote 180/328 | Loss do Lote: 1.5182
Lote 190/328 | Loss do Lote: 1.3809
Lote 200/328 | Loss do Lote: 1.3333
Lote 210/328 | Loss do Lote: 1.3495
Lote 220/328 | Loss do Lote: 1.3519
Lote 230/328 | Loss do Lote: 1.2045
Lote 240/328 | Loss do Lote: 1.2053
Lote 250/328 | Loss do Lote: 1.1048
Lote 260/328 | Loss do Lote: 1.0814
Lote 270/328 | Loss do Lo